# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** *Magic: The Gathering Comprehensive Rules* (effective 19 June 2026), a long rulebook full of exceptions, negations and cross-references.

**Two pipelines over the exact same chunk corpus**

| | Retrieval | Chunks given to the LLM |
|---|---|---|
| Pipeline 1 (baseline) | Bi-Encoder (`bge-small-en-v1.5`), cosine top-10 | top 3 by cosine |
| Pipeline 2 (re-ranking) | the *same* top-10 | top 3 by Cross-Encoder (`ms-marco-MiniLM-L-6-v2`) score |

The LLM is Gemini, with the identical prompt template and temperature 0 for both pipelines.
I keep `GEMINI_API_KEY` in a `.env` file (or the environment) before running. Gemini answers are cached in
`results/llm_cache.json`, so re-running the notebook does not call the API again (delete the file to regenerate).

**Sections:** 1 chunking · 2 index · 3 pipelines · 4 query set and retrieval metrics · 5 answers (Answer 1 vs Answer 2) · 6 latency · 7 scaling · 8 observations

In [1]:
import os, re, time, json, getpass, hashlib
import numpy as np, pandas as pd, faiss, torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
PDF_PATH   = "data/MagicCompRules_20260619.pdf"
BI_MODEL   = "BAAI/bge-small-en-v1.5"                  # Bi-Encoder (dense retriever)
CE_MODEL   = "cross-encoder/ms-marco-MiniLM-L-6-v2"    # Cross-Encoder (re-ranker)
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash-lite")
TOP_K, TOP_N = 10, 3                                   # retrieve 10, give 3 to the LLM
# bge models expect this instruction in front of *queries* (not documents)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
os.makedirs(".cache", exist_ok=True); os.makedirs("results", exist_ok=True)
torch.set_num_threads(os.cpu_count())

/home/vikash/Desktop/git_folders/LMA_Assignment_1/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load and chunk the PDF
Chunks follow the rulebook's own structure: a numbered rule (e.g. `702.3.`) with its sub-rules (`702.3a`, `702.3b`) stays together, small neighbouring rules are merged, and long ones are split at about 1500 characters. Each chunk keeps its section heading so the text is understandable on its own.

In [2]:
reader = PdfReader(PDF_PATH)
raw = "\n".join(p.extract_text() for p in reader.pages)

def build_chunks(text, max_chars=1500, min_chars=400):
    # The table of contents repeats the headings, so take the LAST "1. Game Concepts" (start of the
    # rules) and the LAST "Glossary" (end of the rules) and keep only what lies between them.
    start = [m.start() for m in re.finditer(r"^1\. Game Concepts", text, re.M)][-1]
    end   = [m.start() for m in re.finditer(r"^Glossary", text, re.M)][-1]
    body = re.sub(r"[ \t]+", " ", text[start:end])
    body = re.sub(r"\s*\n\s*", "\n", body)

    # PDF lines are hard-wrapped: glue a line to the previous one unless it begins a new rule/heading
    new_item = re.compile(r"^(\d{3}\.\d+[a-z]?\.?\s|\d{3}\.\s|\d\.\s[A-Z])")
    paras = []
    for line in body.split("\n"):
        if new_item.match(line) or not paras: paras.append(line)
        else: paras[-1] += " " + line

    chunks, heading, cur = [], "", []
    def flush():
        nonlocal cur
        if cur:
            chunks.append({"rule": cur[0].split()[0].rstrip("."), "heading": heading, "text": "\n".join(cur)})
            cur = []
    for p in paras:
        if re.match(r"^\d{3}\. \S", p):            # section heading like "702. Keyword Abilities"
            flush(); heading = p; continue
        if re.match(r"^\d\. [A-Z]", p):              # chapter heading like "7. Additional Rules"
            flush(); continue
        size = sum(len(x) for x in cur)
        if re.match(r"^\d{3}\.\d+\.\s", p) and size >= min_chars: flush()   # new top-level rule
        elif cur and size + len(p) > max_chars: flush()                          # too long: split
        cur.append(p)
    flush()
    for i, c in enumerate(chunks): c["id"] = i
    return chunks

chunks = build_chunks(raw)
# What is embedded / shown to the LLM: heading + text
docs = [f"{c['heading']}\n{c['text']}" for c in chunks]
lens = [len(c["text"]) for c in chunks]
print(f"{len(reader.pages)} pages -> {len(chunks)} chunks | chars/chunk: mean={np.mean(lens):.0f}, max={max(lens)}")
print(docs[557][:400])

311 pages -> 971 chunks | chars/chunk: mean=848, max=2879
702. Keyword Abilities
702.8. Flash
702.8a Flash is a static ability that functions in any zone from which you could play the card it ’s on. “Flash” means “You may play this card any time you could cast an instant.”
702.8b Multiple instances of flash on the same object are redundant.
702.9. Flying
702.9a Flying is an evasion ability.
702.9b A creature with flying can’t be blocked except by creatur


### 1.1 Do the chunks fit the models' input limit?
Both models read at most 512 tokens; anything longer is silently cut off. This counts how many chunks are affected.

In [3]:
from transformers import AutoTokenizer
n_tokens = [len(ids) for ids in AutoTokenizer.from_pretrained(BI_MODEL)(docs)["input_ids"]]
print(f"tokens per chunk: median {int(np.median(n_tokens))}, max {max(n_tokens)}")
print(f"chunks longer than 512 tokens (truncated by both models): {sum(n > 512 for n in n_tokens)} of {len(docs)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (554 > 512). Running this sequence through the model will result in indexing errors


tokens per chunk: median 187, max 952
chunks longer than 512 tokens (truncated by both models): 5 of 971


## 2. Index the chunks (Bi-Encoder + FAISS)
Vectors are L2-normalised, so the inner product equals cosine similarity (`IndexFlatIP` = exact search).

In [4]:
bi_encoder = SentenceTransformer(BI_MODEL)
cross_encoder = CrossEncoder(CE_MODEL)

cache = f".cache/emb_{BI_MODEL.split('/')[-1]}_{len(chunks)}.npy"
if os.path.exists(cache):
    emb = np.load(cache)
else:
    t0 = time.perf_counter()
    emb = bi_encoder.encode(docs, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    print(f"Embedded {len(docs)} chunks in {time.perf_counter()-t0:.1f}s")
    np.save(cache, emb)
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb.astype("float32"))
print("Index size:", index.ntotal, "| dim:", emb.shape[1])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4133.12it/s]


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3809.44it/s]

Index size: 971 | dim: 384


## 3. The two pipelines
Both pipelines share the same prompt builder and the same LLM call, so the **only** difference is which 3 chunks are chosen.

Gemini is called through `generate()`, which caches every answer together with the time of the API call. Only the
**successful** attempt is timed, so the back-off waits after rate-limit errors are not counted as latency.

In [5]:
def retrieve(query, k=TOP_K):
    """Stage 1 (Bi-Encoder): query and chunks are embedded independently; rank by cosine similarity."""
    q = bi_encoder.encode(QUERY_PREFIX + query, normalize_embeddings=True).astype("float32")[None]
    scores, ids = index.search(q, k)
    return list(ids[0]), list(scores[0])

def rerank(query, ids):
    """Stage 2 (Cross-Encoder): score every (query, chunk) pair jointly, then re-sort descending."""
    scores = cross_encoder.predict([(query, docs[i]) for i in ids])
    order = np.argsort(-scores)
    return [ids[j] for j in order], [float(scores[j]) for j in order]

PROMPT = """You are an assistant answering questions about the Magic: The Gathering Comprehensive Rules.
Answer ONLY from the context below. Cite the rule numbers you use. If the context does not contain the answer, say "The provided context does not answer this."

Context:
{context}

Question: {question}

Answer (concise):"""

def build_prompt(question, ids):
    context = "\n\n".join(f"[Chunk {n+1}]\n{docs[i]}" for n, i in enumerate(ids))
    return PROMPT.format(context=context, question=question)

# --- Gemini client ---
api_key = os.environ.get("GEMINI_API_KEY") or getpass.getpass("GEMINI_API_KEY: ")
client = genai.Client(api_key=api_key)

from google.genai import errors
import httpx

LLM_CACHE = "results/llm_cache.json"
llm_cache = json.load(open(LLM_CACHE)) if os.path.exists(LLM_CACHE) else {}

def call_gemini(prompt, retries=6):
    """One real Gemini call -> (answer, seconds). Retries on rate-limit / server errors; only the successful attempt is timed."""
    for attempt in range(retries):
        try:
            t0 = time.perf_counter()
            r = client.models.generate_content(
                model=GEMINI_MODEL, contents=prompt,
                config=types.GenerateContentConfig(temperature=0, thinking_config=types.ThinkingConfig(thinking_level="MINIMAL")))
            return (r.text or "").strip(), time.perf_counter() - t0
        except (errors.APIError, httpx.TransportError) as e:   # rate limit, server busy, or dropped connection
            code = getattr(e, "code", "network")
            if code not in (429, 500, 503, "network") or attempt == retries - 1:
                raise
            wait = 5 * 2 ** attempt
            print(f"  Gemini error {code}: retrying in {wait}s"); time.sleep(wait)

def generate(prompt):
    """Answer from the cache if this exact prompt was answered before, otherwise call Gemini and cache it."""
    key = hashlib.sha256(f"{GEMINI_MODEL}|{prompt}".encode()).hexdigest()
    if key not in llm_cache:
        answer, seconds = call_gemini(prompt)
        llm_cache[key] = {"answer": answer, "seconds": seconds}
        json.dump(llm_cache, open(LLM_CACHE, "w"), indent=1)
        time.sleep(2)                               # stay under the free-tier requests-per-minute limit
    return llm_cache[key]["answer"], llm_cache[key]["seconds"]

def pipeline1(question):
    t0 = time.perf_counter(); ids, sc = retrieve(question); t_ret = time.perf_counter() - t0
    top3 = ids[:TOP_N]
    ans, t_llm = generate(build_prompt(question, top3))
    return dict(candidates=ids, top3=top3, answer=ans, t_retrieve=t_ret, t_rerank=0.0, t_llm=t_llm)

def pipeline2(question, baseline):
    """Re-rank the top-10 candidates returned by Pipeline 1's retrieval (no second search)."""
    ids, t_ret = baseline["candidates"], baseline["t_retrieve"]
    t0 = time.perf_counter(); new_ids, ce_sc = rerank(question, ids); t_rr = time.perf_counter() - t0
    top3 = new_ids[:TOP_N]
    ans, t_llm = generate(build_prompt(question, top3))
    return dict(candidates=new_ids, top3=top3, answer=ans, t_retrieve=t_ret, t_rerank=t_rr, t_llm=t_llm, ce_scores=ce_sc)

## 4. Query set and retrieval metrics
**49 candidate queries** written in player language, covering every failure type in the assignment (negation / constraint,
exception / corner case, keyword trap, buried chunk) plus control queries. Each query has a hand-checked **gold** list: regexes
for the rule line(s) that answer it. When the same rule is stated in more than one place (e.g. "ten or more poison counters"
appears in 104.3d, 122.1f and 704.5c), every chunk that answers the question on its own counts as gold.

The gold rules were found by searching the rulebook text, not by looking at what either retriever returned.
Q1-Q8 are the queries from the first round; Q9-Q49 were added to test the pipelines on a wider, less hand-picked set.

In [6]:
NEG, EXC, KEY, BUR, CTL = "Negation / constraint", "Exception / corner case", "Keyword trap", "Buried chunk", "Control"
TESTS = [
 # id, type, question, gold regexes (any matching chunk answers the question), expected answer (written by hand)
 dict(id="Q1", type=NEG, q="Can a creature with defender attack?", gold=[r"702\.3b"],
      expected="No. A creature with defender can't attack (702.3b)."),
 dict(id="Q2", type=EXC, q="Can a creature with menace be blocked by exactly one creature?", gold=[r"702\.111b"],
      expected="No. A creature with menace can't be blocked except by two or more creatures (702.111b)."),
 dict(id="Q3", type=KEY, q="Can a creature with vigilance still attack without tapping?", gold=[r"702\.20b"],
      expected="Yes. Attacking doesn't cause a creature with vigilance to tap (702.20b)."),
 dict(id="Q4", type=KEY, q="How does the fight keyword action work?", gold=[r"701\.14a"],
      expected="Two creatures each deal damage equal to their power to the other (701.14a)."),
 dict(id="Q5", type=BUR, q="What is the cleanup step used for?", gold=[r"514\.1\. "],
      expected="Active player discards down to maximum hand size, then damage wears off and 'until end of turn' effects end (514)."),
 dict(id="Q6", type=EXC, q="What happens if I try to cast a spell without paying its mana cost?", gold=[r"118\.9\. "],
      expected="It is an alternative cost: no other alternative cost can be used, additional costs still apply, mana cost/value unchanged (118.9)."),
 dict(id="Q7", type=CTL, q="Is it possible to target a creature with hexproof using my own spell?", gold=[r"702\.11b"],
      expected="Yes. Hexproof only stops spells/abilities controlled by opponents (702.11b)."),
 dict(id="Q8", type=CTL, q="Can a creature with haste attack the turn it comes into play?", gold=[r"702\.10b"],
      expected="Yes. Haste lets it attack and use {T} abilities despite not being under control since the turn began (702.10b)."),
 dict(id="Q9", type=CTL, q="How much damage does a blocked creature with trample deal to the defending player?", gold=[r"702\.19b"],
      expected="Lethal damage must be assigned to all blockers first; the excess can be assigned to the player (702.19b)."),
 dict(id="Q10", type=CTL, q="Does a creature with lifelink make its controller gain life?", gold=[r"702\.15b", r"120\.3f"],
      expected="Yes. Damage dealt by a lifelink source makes its controller gain that much life (702.15b)."),
 dict(id="Q11", type=NEG, q="Is an indestructible creature destroyed when it is dealt lethal damage?", gold=[r"702\.12b"],
      expected="No. Indestructible permanents can't be destroyed and aren't destroyed by lethal damage (702.12b)."),
 dict(id="Q12", type=KEY, q="If a deathtouch creature deals just 1 damage to a big creature, what happens?", gold=[r"702\.2b"],
      expected="It is destroyed as a state-based action: any damage from a deathtouch source is enough (702.2b)."),
 dict(id="Q13", type=EXC, q="What happens if I control two legendary creatures with the same name?", gold=[r"704\.5j"],
      expected="Legend rule: the controller chooses one, the rest are put into their owners' graveyards (704.5j)."),
 dict(id="Q14", type=BUR, q="How many poison counters does it take to lose the game?", gold=[r"704\.5c", r"104\.3d", r"122\.1f"],
      expected="Ten or more poison counters (704.5c, 104.3d)."),
 dict(id="Q15", type=EXC, q="What happens if a player has to draw from an empty library?", gold=[r"704\.5b", r"121\.4\. "],
      expected="That player loses the game the next time state-based actions are checked (704.5b, 121.4)."),
 dict(id="Q16", type=EXC, q="Does the starting player draw a card on their first turn?", gold=[r"103\.8a"],
      expected="No. In a two-player game the starting player skips the draw step of their first turn (103.8a)."),
 dict(id="Q17", type=NEG, q="Can I play a land during my opponent's turn?", gold=[r"305\.3\. "],
      expected="No. A player can't play a land if it isn't their turn (305.3)."),
 dict(id="Q18", type=NEG, q="How many lands can a player play each turn?", gold=[r"305\.2\. "],
      expected="Normally one, unless effects allow more (305.2)."),
 dict(id="Q19", type=NEG, q="Can a sorcery be cast during combat?", gold=[r"307\.1\. ", r"117\.1a"],
      expected="No. Sorceries only in the caster's own main phase with an empty stack (307.1)."),
 dict(id="Q20", type=NEG, q="Can a player respond to a mana ability?", gold=[r"605\.3b"],
      expected="No. Mana abilities don't use the stack and can't be responded to (605.3b)."),
 dict(id="Q21", type=KEY, q="Can a creature use a tap ability the turn it comes under my control?", gold=[r"302\.6\. "],
      expected="No, unless it has haste: it must have been under its controller's control since their most recent turn began (302.6)."),
 dict(id="Q22", type=CTL, q="What is the maximum hand size?", gold=[r"402\.2\. "],
      expected="Normally seven; excess cards are discarded in the cleanup step (402.2)."),
 dict(id="Q23", type=BUR, q="How much commander damage makes a player lose?", gold=[r"903\.10a"],
      expected="21 or more combat damage from the same commander (903.10a)."),
 dict(id="Q24", type=NEG, q="Can a creature with protection from red be blocked by red creatures?", gold=[r"702\.16f"],
      expected="No. Attacking creatures with protection can't be blocked by creatures with that quality (702.16f)."),
 dict(id="Q25", type=KEY, q="Can I target my own creature that has shroud?", gold=[r"702\.18a"],
      expected="No. Shroud stops targeting by all spells and abilities, including the controller's own (702.18a)."),
 dict(id="Q26", type=NEG, q="Which creatures are able to block a creature with flying?", gold=[r"702\.9b", r"702\.17b"],
      expected="Only creatures with flying and/or reach (702.9b)."),
 dict(id="Q27", type=KEY, q="If a token creature dies, does it stay in the graveyard?", gold=[r"111\.7\. ", r"111\.8\. ", r"704\.5d"],
      expected="No. A token in a zone other than the battlefield ceases to exist (111.7, 704.5d)."),
 dict(id="Q28", type=BUR, q="What happens to a creature with 0 toughness?", gold=[r"704\.5f"],
      expected="It is put into its owner's graveyard as a state-based action; regeneration can't replace it (704.5f)."),
 dict(id="Q29", type=KEY, q="What does it mean when a creature dies?", gold=[r"700\.4\. "],
      expected="'Dies' means it is put into a graveyard from the battlefield (700.4)."),
 dict(id="Q30", type=NEG, q="Can equipment be moved to another creature during combat?", gold=[r"702\.6a"],
      expected="No. Equip can only be activated as a sorcery (main phase, empty stack) (702.6a)."),
 dict(id="Q31", type=EXC, q="If the creature blocking my attacker is removed, does my attacker still deal damage?", gold=[r"510\.1c"],
      expected="No. A blocked creature with no blockers assigns no combat damage (unless e.g. trample) (510.1c)."),
 dict(id="Q32", type=NEG, q="Can a creature with intimidate be blocked by a creature of a different color?", gold=[r"702\.13b"],
      expected="Only if it is an artifact creature; otherwise blockers must share a color (702.13b)."),
 dict(id="Q33", type=CTL, q="What happens when ward triggers?", gold=[r"702\.21a"],
      expected="The opponent's spell or ability that targeted it is countered unless that player pays the ward cost (702.21a)."),
 dict(id="Q34", type=NEG, q="Can a player with 0 life keep playing?", gold=[r"704\.5a", r"104\.3b", r"119\.6\. "],
      expected="No. A player with 0 or less life loses the game (704.5a)."),
 dict(id="Q35", type=CTL, q="Can a player concede at any time?", gold=[r"104\.3a", r"405\.6f", r"723\.6", r".*concede the game at any time"],
      expected="Yes. A player can concede at any time and loses the game (104.3a)."),
 dict(id="Q36", type=BUR, q="What happens to a creature that is removed from combat?", gold=[r"506\.4\. "],
      expected="It stops being an attacking/blocking creature (506.4)."),
 dict(id="Q37", type=KEY, q="How does double strike change the combat damage step?", gold=[r"702\.4b"],
      expected="There are two combat damage steps; double strike creatures deal damage in both (702.4b)."),
 dict(id="Q38", type=KEY, q="How many cards does each player draw at the start of the game?", gold=[r"103\.5\. "],
      expected="Their starting hand size, normally seven (103.5)."),
 dict(id="Q39", type=NEG, q="Can a planeswalker's loyalty ability be activated twice in one turn?", gold=[r"113\.5\. ", r"606\.3\. "],
      expected="No. Only one loyalty ability of a permanent per turn, main phase, empty stack (606.3)."),
 dict(id="Q40", type=BUR, q="What happens to a planeswalker with no loyalty left?", gold=[r"704\.5i", r"122\.1e", r"306\.9\. "],
      expected="It is put into its owner's graveyard (704.5i)."),
 dict(id="Q41", type=EXC, q="What happens if a creature has both a +1/+1 counter and a -1/-1 counter?", gold=[r"704\.5q", r"122\.3\. "],
      expected="N of each are removed, where N is the smaller count (704.5q)."),
 dict(id="Q42", type=NEG, q="Am I allowed to look through my library during the game?", gold=[r"401\.2\. "],
      expected="No. Players can't look at or change the order of cards in a library (401.2)."),
 dict(id="Q43", type=CTL, q="Can I look at the cards in my opponent's graveyard?", gold=[r"404\.2\. "],
      expected="Yes. Any graveyard can be examined at any time (404.2)."),
 dict(id="Q44", type=KEY, q="What does mill mean?", gold=[r"701\.17a"],
      expected="Put that many cards from the top of the player's library into their graveyard (701.17a)."),
 dict(id="Q45", type=EXC, q="Does a spell resolve if all its targets have become illegal?", gold=[r"608\.2b"],
      expected="No. If all its targets are illegal, it doesn't resolve (608.2b)."),
 dict(id="Q46", type=CTL, q="In what order do spells on the stack resolve?", gold=[r"405\.5\. "],
      expected="The top (last-added) spell or ability resolves first (405.5)."),
 dict(id="Q47", type=NEG, q="Can a tapped creature block?", gold=[r"509\.1a"],
      expected="No. Chosen blockers must be untapped (509.1a)."),
 dict(id="Q48", type=CTL, q="When can a player cast an instant?", gold=[r"117\.1a", r"304\.1\. "],
      expected="Any time they have priority (117.1a)."),
 dict(id="Q49", type=EXC, q="Does an Aura stay on the battlefield when the enchanted creature leaves?", gold=[r"704\.5m", r"303\.4c"],
      expected="No. An Aura not attached to a legal object is put into its owner's graveyard (704.5m)."),
]

def gold_chunk_ids(patterns):
    # anchor at line start so a cross-reference such as "see rule 514.1" does not count as the gold chunk
    return {c["id"] for c in chunks for p in patterns if re.search("(?m)^" + p, c["text"])}

def rank_of(ids, gold_ids):
    for r, i in enumerate(ids, 1):
        if i in gold_ids: return r
    return None   # not in the retrieved top 10

for t in TESTS:
    t["gold_ids"] = gold_chunk_ids(t["gold"])
    assert t["gold_ids"], f"gold pattern for {t['id']} matches no chunk"
print(len(TESTS), "queries |", pd.Series([t["type"] for t in TESTS]).value_counts().to_dict())

49 queries | {'Negation / constraint': 14, 'Keyword trap': 10, 'Control': 10, 'Exception / corner case': 9, 'Buried chunk': 6}


### 4a. Retrieval only (no LLM): where does the gold chunk land?
| Outcome | Meaning |
|---|---|
| fixed by re-rank | gold at rank 4-10 for the baseline, in the top 3 after re-ranking |
| both ok | gold in the top 3 both times |
| hurt by re-rank | gold in the top 3 for the baseline, pushed out by re-ranking |
| both fail | gold in the top 10 but outside the top 3 both times |
| not in top 10 | gold not retrieved at all: the re-ranker only reorders the 10 candidates, so it cannot help |

In [7]:
def outcome(b, r):
    if b is None:                return "not in top 10"
    if b > TOP_N and r <= TOP_N: return "fixed by re-rank"
    if b <= TOP_N and r <= TOP_N: return "both ok"
    if b <= TOP_N:               return "hurt by re-rank"
    return "both fail"

rows = []
for t in TESTS:
    ids, _ = retrieve(t["q"]); rr_ids, _ = rerank(t["q"], ids)
    b, r = rank_of(ids, t["gold_ids"]), rank_of(rr_ids, t["gold_ids"])
    t["outcome"] = outcome(b, r)
    rows.append(dict(id=t["id"], type=t["type"], baseline_rank=b, reranked_rank=r, outcome=t["outcome"]))
rank_df = pd.DataFrame(rows).astype({"baseline_rank": "Int64", "reranked_rank": "Int64"})
pd.set_option("display.max_rows", 100)
rank_df

,id,type,baseline_rank,reranked_rank,outcome
0,Q1,Negation / constraint,5,1,fixed by re-rank
1,Q2,Exception / corner case,5,1,fixed by re-rank
2,Q3,Keyword trap,4,1,fixed by re-rank
3,Q4,Keyword trap,4,1,fixed by re-rank
4,Q5,Buried chunk,10,1,fixed by re-rank
5,Q6,Exception / corner case,4,2,fixed by re-rank
6,Q7,Control,1,1,both ok
7,Q8,Control,1,1,both ok
8,Q9,Control,1,1,both ok
9,Q10,Control,1,1,both ok


In [8]:
print(rank_df["outcome"].value_counts().to_string(), "\n")
pd.crosstab(rank_df["type"], rank_df["outcome"])

outcome
both ok             31
fixed by re-rank    10
not in top 10        6
hurt by re-rank      2 



outcome,both ok,fixed by re-rank,hurt by re-rank,not in top 10
type,,,,
Buried chunk,4,1,0,1
Control,8,1,1,0
Exception / corner case,4,3,1,1
Keyword trap,7,3,0,0
Negation / constraint,8,2,0,4


### 4b. Retrieval metrics on all 49 queries
- **Hit@3**: the gold chunk is among the 3 chunks the LLM sees.
- **MRR@10**: mean of 1/rank of the gold chunk (0 if it is not in the top 10).
- **Recall@10**: the gold chunk is in the top 10. This is the same for both pipelines, because re-ranking only reorders the same 10.

In [9]:
def retrieval_metrics(ranks):
    ranks = [None if pd.isna(r) else int(r) for r in ranks]
    return {"Hit@3": np.mean([r is not None and r <= TOP_N for r in ranks]),
            "MRR@10": np.mean([1 / r if r else 0.0 for r in ranks]),
            "Recall@10": np.mean([r is not None for r in ranks]),
            "mean gold rank (when in top 10)": np.mean([r for r in ranks if r])}

metrics_df = pd.DataFrame({"Pipeline 1 (baseline)": retrieval_metrics(rank_df["baseline_rank"]),
                           "Pipeline 2 (re-ranked)": retrieval_metrics(rank_df["reranked_rank"])}).T.round(3)
metrics_df

,Hit@3,MRR@10,Recall@10,mean gold rank (when in top 10)
Pipeline 1 (baseline),0.673,0.539,0.878,2.605
Pipeline 2 (re-ranked),0.837,0.710,0.878,1.535


In [10]:
# Same metrics per failure type
per_type = {}
for typ, grp in rank_df.groupby("type"):
    b, r = retrieval_metrics(grp["baseline_rank"]), retrieval_metrics(grp["reranked_rank"])
    per_type[typ] = {"queries": len(grp), "Hit@3 P1": b["Hit@3"], "Hit@3 P2": r["Hit@3"], "MRR@10 P1": b["MRR@10"], "MRR@10 P2": r["MRR@10"]}
pd.DataFrame(per_type).T.round(2)

,queries,Hit@3 P1,Hit@3 P2,MRR@10 P1,MRR@10 P2
Buried chunk,6.0,0.67,0.83,0.43,0.56
Control,10.0,0.90,0.90,0.82,0.87
Exception / corner case,9.0,0.56,0.78,0.40,0.69
Keyword trap,10.0,0.70,1.00,0.58,0.83
Negation / constraint,14.0,0.57,0.71,0.45,0.58


## 5. Answer 1 vs. Answer 2
Gemini answers are generated for a **final set of 23 queries**, chosen *before* any answer was generated:
every query where re-ranking changed whether the gold chunk reaches the top 3 (fixed or hurt), the queries whose gold chunk
was not retrieved at all (to show the limit of re-ranking), and four controls.

**Label audit after reading the answers.** Two answers cited a rule that also answers the question but was missing from the gold list:
Q40 (306.9 also says a planeswalker with 0 loyalty goes to the graveyard) and Q49 (303.4c: an Aura goes to the graveyard when the object it
was attached to no longer exists). Both were added to the gold lists. As a result Q40 moved from *fixed by re-rank* to *both ok*, and Q49 from
*not in top 10* to *hurt by re-rank*. The prompts, and therefore the answers, did not change.

In [11]:
FINAL = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q11", "Q13", "Q27", "Q33",   # fixed by re-rank
         "Q46", "Q49",                                                      # hurt by re-rank
         "Q24", "Q26", "Q28", "Q30", "Q41", "Q47",                          # gold not in the top 10
         "Q7", "Q8", "Q9", "Q40", "Q45"]                                    # both ok (controls)
by_id = {t["id"]: t for t in TESTS}
final_tests = [by_id[i] for i in FINAL]
print(pd.Series([t["outcome"] for t in final_tests]).value_counts().to_string())

fixed by re-rank    10
not in top 10        6
both ok              5
hurt by re-rank      2


In [12]:
results = {}
for t in final_tests:
    p1 = pipeline1(t["q"])                       # Pipeline 1: retrieve top 10, answer from the top 3
    results[t["id"]] = (p1, pipeline2(t["q"], p1))   # Pipeline 2: re-rank Pipeline 1's top 10, answer from the new top 3

def describe(ids, gold): return ", ".join(chunks[i]["rule"] + ("*" if i in gold else "") for i in ids)
for t in final_tests:
    p1, p2 = results[t["id"]]
    print("=" * 100); print(f"{t['id']} [{t['type']} | {t['outcome']}] {t['q']}")
    print(f"  Expected : {t['expected']}")
    print(f"  Top-3 baseline : {describe(p1['top3'], t['gold_ids'])}     (* = gold chunk)")
    print(f"  Top-3 reranked : {describe(p2['top3'], t['gold_ids'])}")
    print(f"  ANSWER 1 (baseline): {p1['answer']}")
    print(f"  ANSWER 2 (rerank)  : {p2['answer']}")
json.dump({k: [{kk: vv for kk, vv in p.items() if kk != 'ce_scores'} for p in v] for k, v in results.items()},
          open("results/answers.json", "w"), indent=1, default=int)

Q1 [Negation / constraint | fixed by re-rank] Can a creature with defender attack?
  Expected : No. A creature with defender can't attack (702.3b).
  Top-3 baseline : 802.2a, 310.5, 508.5     (* = gold chunk)
  Top-3 reranked : 702.3*, 302.5, 802.2a
  ANSWER 1 (baseline): The provided context does not answer this.
  ANSWER 2 (rerank)  : No, a creature with defender can't attack (702.3b).
Q2 [Exception / corner case | fixed by re-rank] Can a creature with menace be blocked by exactly one creature?
  Expected : No. A creature with menace can't be blocked except by two or more creatures (702.111b).
  Top-3 baseline : 702.22h, 302.5, 805.10c     (* = gold chunk)
  Top-3 reranked : 702.110*, 509.1c, 510.1c
  ANSWER 1 (baseline): The provided context does not answer this.
  ANSWER 2 (rerank)  : No. A creature with menace can't be blocked except by two or more creatures (Rule 702.111b).
Q3 [Keyword trap | fixed by re-rank] Can a creature with vigilance still attack without tapping?
  Expected

### 5a. Answer labels
Each answer was compared with the expected answer and with the 3 chunks it was given (I drafted the labels with an AI assistant and then checked them myself):

| Label | Meaning |
|---|---|
| correct | matches the expected answer, including the condition that decides it |
| partial | right direction, but misses an essential clause or condition |
| wrong | states the wrong rule or the opposite answer |
| hallucinated | claims something none of its 3 chunks supports |
| refused | "The provided context does not answer this." |

In [13]:
LABELS = {   # id: (Answer 1 label, Answer 2 label, note)
 "Q1":  ("refused", "correct", ""),
 "Q2":  ("refused", "correct", "Answer 2 cites the correct rule number 702.111b."),
 "Q3":  ("refused", "correct", ""),
 "Q4":  ("refused", "correct", "Answer 2 lists all four fight sub-rules."),
 "Q5":  ("refused", "correct", ""),
 "Q6":  ("wrong", "partial", "A1 explains it through the unpayable-cost rule (118.6a). A2 names it an alternative cost (118.9) but omits that additional costs still apply and mana value is unchanged."),
 "Q11": ("refused", "correct", ""),
 "Q13": ("refused", "correct", ""),
 "Q27": ("refused", "correct", ""),
 "Q33": ("refused", "correct", ""),
 "Q46": ("correct", "partial", "A1 says the top (last-added) object resolves (405.5). A2 only says the top object resolves (608.1) and never states last-in, first-out."),
 "Q49": ("partial", "refused", "A1 gives the right answer from 303.4c, but quotes its 'player has left the game' clause instead of 'the object no longer exists'."),
 "Q24": ("refused", "refused", "Gold 702.16f not in the top 10."),
 "Q26": ("refused", "refused", "Gold 702.9b not in the top 10."),
 "Q28": ("refused", "refused", "A2 only restates 208.5 and then says the context does not answer."),
 "Q30": ("refused", "refused", "Gold 702.6a not in the top 10."),
 "Q41": ("refused", "refused", "Gold 704.5q / 122.3 not in the top 10."),
 "Q47": ("refused", "refused", "Gold 509.1a not in the top 10."),
 "Q7":  ("correct", "correct", ""),
 "Q8":  ("correct", "correct", ""),
 "Q9":  ("correct", "correct", ""),
 "Q40": ("correct", "correct", "Both answer from 306.9 / 122.1e."),
 "Q45": ("correct", "correct", ""),
}
lab = pd.DataFrame([(k, by_id[k]["outcome"], *LABELS[k]) for k in FINAL], columns=["id", "outcome", "Answer 1", "Answer 2", "note"])
display(pd.DataFrame({"Answer 1 (baseline)": lab["Answer 1"].value_counts(),
                      "Answer 2 (re-ranked)": lab["Answer 2"].value_counts()}).fillna(0).astype(int))
pd.set_option("display.max_colwidth", 140)
lab

,Answer 1 (baseline),Answer 2 (re-ranked)
correct,6,14
partial,1,2
refused,15,7
wrong,1,0


,id,outcome,Answer 1,Answer 2,note
0,Q1,fixed by re-rank,refused,correct,
1,Q2,fixed by re-rank,refused,correct,Answer 2 cites the correct rule number 702.111b.
2,Q3,fixed by re-rank,refused,correct,
3,Q4,fixed by re-rank,refused,correct,Answer 2 lists all four fight sub-rules.
4,Q5,fixed by re-rank,refused,correct,
5,Q6,fixed by re-rank,wrong,partial,A1 explains it through the unpayable-cost rule (118.6a). A2 names it an alternative cost (118.9) but omits that additional costs still a...
6,Q11,fixed by re-rank,refused,correct,
7,Q13,fixed by re-rank,refused,correct,
8,Q27,fixed by re-rank,refused,correct,
9,Q33,fixed by re-rank,refused,correct,


### 5b. Full top-10 before vs. after re-ranking (rank shift)
`*` marks the gold chunk. Shown for every final query where the outcome is not "both ok".

In [14]:
for t in final_tests:
    if t["outcome"] == "both ok": continue
    ids, _ = retrieve(t["q"]); rr_ids, _ = rerank(t["q"], ids)
    b, r = rank_of(ids, t["gold_ids"]), rank_of(rr_ids, t["gold_ids"])
    print(f"{t['id']} [{t['outcome']}] {t['q']}   gold rank: #{b} -> #{r}")
    print(f"  baseline : {[chunks[i]['rule'] + ('*' if i in t['gold_ids'] else '') for i in ids]}")
    print(f"  reranked : {[chunks[i]['rule'] + ('*' if i in t['gold_ids'] else '') for i in rr_ids]}\n")

Q1 [fixed by re-rank] Can a creature with defender attack?   gold rank: #5 -> #1
  baseline : ['802.2a', '310.5', '508.5', '805.10e', '702.3*', '302.5', '805.10c', '508.3f', '310.10', '508.1d']
  reranked : ['702.3*', '302.5', '802.2a', '508.5', '805.10e', '508.1d', '508.3f', '805.10c', '310.5', '310.10']



Q2 [fixed by re-rank] Can a creature with menace be blocked by exactly one creature?   gold rank: #5 -> #1
  baseline : ['702.22h', '302.5', '805.10c', '509.1c', '702.110*', '509.3c', '510.1c', '702.19d', '509.3e', '702.22k']
  reranked : ['702.110*', '509.1c', '510.1c', '702.22h', '509.3c', '702.22k', '509.3e', '805.10c', '702.19d', '302.5']



Q3 [fixed by re-rank] Can a creature with vigilance still attack without tapping?   gold rank: #4 -> #1
  baseline : ['302.5', '509.3g', '506.5', '702.20*', '509.3c', '506.3', '702.31', '506.3f', '702.19b', '508.3f']
  reranked : ['702.20*', '302.5', '702.31', '506.3f', '506.5', '508.3f', '509.3g', '506.3', '702.19b', '509.3c']



Q4 [fixed by re-rank] How does the fight keyword action work?   gold rank: #4 -> #1
  baseline : ['701.1', '701.23f', '701.45', '701.13*', '702.7', '702.1c', '702.3', '701.54e', '702.189', '704.4']
  reranked : ['701.13*', '701.45', '701.23f', '701.1', '701.54e', '702.7', '702.3', '702.1c', '702.189', '704.4']



Q5 [fixed by re-rank] What is the cleanup step used for?   gold rank: #10 -> #1
  baseline : ['512.1', '514.3', '724.1f', '513.2', '703.4m', '503.1', '724.1', '500.3', '120.6', '514.1*']
  reranked : ['514.1*', '514.3', '724.1f', '120.6', '703.4m', '512.1', '500.3', '513.2', '724.1', '503.1']



...

Q26 [not in top 10] Which creatures are able to block a creature with flying?   gold rank: #None -> #None
  baseline : ['302.5', '702.22h', '509.3g', '805.10c', '509.3c', '509.3e', '508.3f', '702.19d', '509.1', '702.19b']
  reranked : ['702.22h', '509.1', '302.5', '702.19b', '702.19d', '509.3c', '509.3e', '805.10c', '509.3g', '508.3f']



Q28 [not in top 10] What happens to a creature with 0 toughness?   gold rank: #None -> #None
  baseline : ['208.5', '208.2b', '302.4', '208.1', '208.3', '701.10', '613.4d', '208.4', '302.7', '107.1b']
  reranked : ['208.5', '613.4d', '701.10', '208.2b', '208.1', '107.1b', '302.7', '208.4', '302.4', '208.3']



Q30 [not in top 10] Can equipment be moved to another creature during combat?   gold rank: #None -> #None
  baseline : ['301.5c', '701.3d', '707.5', '400.6', '301.4', '701.3', '701.13', '702.22k', '804.1', '301.5f']
  reranked : ['701.3d', '301.5c', '301.4', '701.13', '701.3', '702.22k', '301.5f', '400.6', '707.5', '804.1']



Q41 [not in top 10] What happens if a creature has both a +1/+1 counter and a -1/-1 counter?   gold rank: #None -> #None
  baseline : ['122.1', '122.1d', '712.21c', '702.19c', '702.19d', '107.1b', '701.10', '509.3e', '701.10g', '121.7']
  reranked : ['712.21c', '122.1', '701.10', '701.10g', '107.1b', '122.1d', '509.3e', '702.19d', '702.19c', '121.7']



Q47 [not in top 10] Can a tapped creature block?   gold rank: #None -> #None
  baseline : ['302.5', '509.3c', '509.3e', '509.3g', '702.22h', '509.1c', '805.10c', '702.110', '702.22k', '509.1d']
  reranked : ['302.5', '702.22h', '702.22k', '509.1c', '509.3c', '509.3g', '509.3e', '805.10c', '509.1d', '702.110']



## 6. Latency profiling
Retrieval stages are timed over 10 repeats of the 23 final queries after a warm-up pass. The LLM time is the single
successful Gemini call per query and pipeline (retry waits excluded); it is network-bound and does not depend on the pipeline.

In [15]:
N_RUNS = 10
def timed(fn, *a):
    t0 = time.perf_counter(); out = fn(*a); return out, time.perf_counter() - t0

for t in final_tests:                       # warm-up pass (first calls load kernels / caches)
    rerank(t["q"], retrieve(t["q"])[0])

ret_t, rr_t = [], []
for _ in range(N_RUNS):
    for t in final_tests:
        (ids, _), a = timed(retrieve, t["q"]);  ret_t.append(a)
        _, b = timed(rerank, t["q"], ids);      rr_t.append(b)
ret_t, rr_t = np.array(ret_t), np.array(rr_t)
llm1 = np.array([results[k][0]["t_llm"] for k in results]); llm2 = np.array([results[k][1]["t_llm"] for k in results])

def row(stage, x): return dict(stage=stage, median_ms=np.median(x) * 1e3, mean_ms=np.mean(x) * 1e3, p95_ms=np.percentile(x, 95) * 1e3)
lat = pd.DataFrame([
    row("Pipeline 1: Bi-Encoder retrieval (top-10)", ret_t),
    row("Pipeline 2 extra: Cross-Encoder re-rank (10 pairs)", rr_t),
    row("Pipeline 2 total retrieval (retrieve + re-rank)", ret_t + rr_t),
    row("Gemini generation, Pipeline 1 prompts", llm1),
    row("Gemini generation, Pipeline 2 prompts", llm2),
    row("End-to-end Pipeline 1 (retrieval + LLM)", np.median(ret_t) + llm1),
    row("End-to-end Pipeline 2 (retrieval + re-rank + LLM)", np.median(ret_t + rr_t) + llm2),
]).round(1)
print(f"{len(ret_t)} timed runs per retrieval stage, {len(llm1)} LLM calls per pipeline")
print(f"Re-rank overhead: {np.median(rr_t)*1e3:.0f} ms median = {100*np.median(rr_t)/np.median(np.median(ret_t+rr_t)+llm2):.0f}% of Pipeline 2 end-to-end")
lat

230 timed runs per retrieval stage, 23 LLM calls per pipeline
Re-rank overhead: 682 ms median = 38% of Pipeline 2 end-to-end


,stage,median_ms,mean_ms,p95_ms
0,Pipeline 1: Bi-Encoder retrieval (top-10),27.8,28.3,32.0
1,Pipeline 2 extra: Cross-Encoder re-rank (10 pairs),681.7,744.7,1035.2
2,Pipeline 2 total retrieval (retrieve + re-rank),709.1,773.0,1064.0
3,"Gemini generation, Pipeline 1 prompts",1137.5,1403.2,1703.6
4,"Gemini generation, Pipeline 2 prompts",1073.0,1050.7,1535.2
5,End-to-end Pipeline 1 (retrieval + LLM),1165.3,1431.0,1731.4
6,End-to-end Pipeline 2 (retrieval + re-rank + LLM),1782.1,1759.8,2244.3


## 7. Scaling: why not run the Cross-Encoder over the whole collection?
The Cross-Encoder needs one forward pass per (query, chunk) pair at query time. Here its time is measured for a growing number
of candidates, fitted with a straight line, and compared with the cost of the two-stage design for larger collections.

In [16]:
q = TESTS[0]["q"]
order, _ = retrieve(q, k=len(docs))          # all chunks, best first
ce_ms = {}
for n in [10, 50, 100, 250, 500, len(docs)]:
    reps = 3 if n <= 100 else 1
    ce_ms[n] = np.median([timed(rerank, q, order[:n])[1] for _ in range(reps)]) * 1e3
ms_per_pair = np.polyfit(list(ce_ms), list(ce_ms.values()), 1)[0]
print("Cross-Encoder time (ms) by number of pairs:", {n: round(v) for n, v in ce_ms.items()})
print(f"-> about {ms_per_pair:.1f} ms per (query, chunk) pair, growing linearly")

# Exact vector search time for a large index (random unit vectors, same dimension), to extrapolate the first stage
rng = np.random.default_rng(0)
big = rng.standard_normal((200_000, emb.shape[1])).astype("float32"); big /= np.linalg.norm(big, axis=1, keepdims=True)
big_index = faiss.IndexFlatIP(emb.shape[1]); big_index.add(big)
qv = bi_encoder.encode(QUERY_PREFIX + q, normalize_embeddings=True).astype("float32")[None]
search_ms_per_vec = np.median([timed(big_index.search, qv, TOP_K)[1] for _ in range(5)]) * 1e3 / 200_000
encode_ms = np.median([timed(bi_encoder.encode, QUERY_PREFIX + q)[1] for _ in range(5)]) * 1e3
del big, big_index

pd.DataFrame([{"chunks N": f"{n:,}",
               "Cross-Encoder on all N (s)": ms_per_pair * n / 1e3,
               "two-stage: encode + exact search + CE on 10 (s)": (encode_ms + search_ms_per_vec * n + ce_ms[10]) / 1e3}
              for n in [len(docs), 100_000, 1_000_000, 10_000_000]]).set_index("chunks N").round(2)

Cross-Encoder time (ms) by number of pairs: {10: 687, 50: 2844, 100: 6766, 250: 14896, 500: 26434, 971: 48393}
-> about 49.2 ms per (query, chunk) pair, growing linearly


,Cross-Encoder on all N (s),two-stage: encode + exact search + CE on 10 (s)
chunks N,,
971,47.82,0.71
"100,000",4924.44,0.72
"1,000,000",49244.36,0.85
"10,000,000",492443.64,2.09


### 7a. What does the first stage cost in quality?
For the queries whose gold chunk was **not in the top 10**, the Cross-Encoder scores *all* chunks. If it finds the gold chunk,
the miss is caused by the Bi-Encoder stage; if it also misses, both models fail to connect the query to the rule.

In [17]:
rows = []
for t in TESTS:
    if t["outcome"] != "not in top 10": continue
    all_bi, _ = retrieve(t["q"], k=len(docs))
    all_ce, _ = rerank(t["q"], list(range(len(docs))))
    rows.append(dict(id=t["id"], q=t["q"], bi_rank_all=rank_of(all_bi, t["gold_ids"]), ce_rank_all=rank_of(all_ce, t["gold_ids"])))
pd.DataFrame(rows)

,id,q,bi_rank_all,ce_rank_all
0,Q24,Can a creature with protection from red be blocked by red creatures?,13,3
1,Q26,Which creatures are able to block a creature with flying?,11,1
2,Q28,What happens to a creature with 0 toughness?,33,11
3,Q30,Can equipment be moved to another creature during combat?,25,16
4,Q41,What happens if a creature has both a +1/+1 counter and a -1/-1 counter?,14,1
5,Q47,Can a tapped creature block?,16,27


## 8. Observations
The numbers for the report come from the cells above: retrieval metrics (4a, 4b), the answer comparison and labels (5, 5a),
the rank shifts (5b), the latency table (6) and the scaling measurements (7, 7a).